In [ ]:
from shared_utils.catalog_utils import get_catalog
from shared_utils.rt_dates import DATES
from calitp_data_analysis import sql, gcs_pandas
import pandas as pd
import datetime

g = gcs_pandas.GCSPandas()

## Constants

In [ ]:
# The feed to examine TODO: support multiple
GTFS_DATASET_NAME = "Bay Area 511 SamTrans Schedule"
# Sample dates to look at
dates_values = DATES.values()
print(len(dates_values))

## Get data

In [ ]:
# Get feed key for the target feed:
dates_str = "('" + "', '".join(dates_values) + "')"
feed_keys_to_dates = sql.query_sql(
    f"""
    select 
        tu.base64_url as tu_base64_url,
        sched.feed_key as schedule_feed_key,
        sched.date
    from mart_gtfs.fct_daily_schedule_feeds as sched
    left join mart_gtfs.fct_daily_rt_feed_files as tu
        on sched.feed_key = tu.schedule_feed_key
        and sched.date = tu.date
        and tu.feed_type = 'trip_updates'
    where 
        sched.date in {dates_str} 
        and sched.gtfs_dataset_name = '{GTFS_DATASET_NAME}'
    """,

)
feed_keys_to_dates

In [ ]:
# Count fct_stop_time_metrics rows per trip updates feed per date
# The dates with a significant number of rows are the dates where both TU and VP based stop times are available
tu_urls = feed_keys_to_dates["tu_base64_url"].dropna().unique()
tu_urls_str = "('" + "', '".join(tu_urls) + "')"

stop_time_metrics_counts = sql.query_sql(
    f"""
    select
        base64_url as tu_base64_url,
        service_date,
        count(*) as n_stop_time_metrics
    from mart_gtfs.fct_stop_time_metrics
    where
        service_date in {dates_str}
        and base64_url in {tu_urls_str}
    group by base64_url, service_date
    order by service_date
    """,
)
stop_time_metrics_counts


In [ ]:
# Get the VP based stop times
gtfs_analytics_catalog = get_catalog("gtfs_analytics_data")
stop_time_metrics_counts["vp_uri"] = (
    f"{gtfs_analytics_catalog['rt_stop_times']['dir']}{gtfs_analytics_catalog['rt_stop_times']['stage3']}_" 
    + stop_time_metrics_counts["service_date"].astype(str)
    + ".parquet"
)


In [ ]:
# Outer join VP based and TU based stop times, per date.
# fct_stop_time_metrics has no trip_instance_key, so fct_scheduled_trips bridges
# its (trip_id, schedule_base64_url, service_date) grain to the VP trip_instance_key.
stm_dates = stop_time_metrics_counts["service_date"].astype(str).tolist()
stm_dates_str = "('" + "', '".join(stm_dates) + "')"
stm_urls_str = "('" + "', '".join(stop_time_metrics_counts["tu_base64_url"].unique()) + "')"

# The VP parquets are statewide, so we need this operator's trips to filter them.
# This also keeps trips that VP saw but TU never predicted.
operator_trips = sql.query_sql(
    f"""
    select trip_instance_key, trip_id, service_date
    from mart_gtfs.fct_scheduled_trips
    where
        service_date in {stm_dates_str}
        and name = '{GTFS_DATASET_NAME}'
    """,
)

# One partition-pruned scan for all dates rather than a query per date.
# Literal service_date/base64_url filters on both tables keep BigQuery from
# scanning the full fct_scheduled_trips history (~48GB) via the join predicate.
# Ordered by trip and stop so the frame arrives ready for within-trip lags.
tu_stop_times = sql.query_sql(
    f"""
    with stm as (
        select service_date, schedule_base64_url, trip_id, stop_id, stop_sequence,
               actual_arrival_pacific, n_predictions
        from mart_gtfs.fct_stop_time_metrics
        where
            service_date in {stm_dates_str}
            and base64_url in {stm_urls_str}
    ),
    sched as (
        select trip_instance_key, trip_id, base64_url, service_date
        from mart_gtfs.fct_scheduled_trips
        where
            service_date in {stm_dates_str}
            and name = '{GTFS_DATASET_NAME}'
    )
    select
        sched.trip_instance_key,
        stm.service_date,
        stm.stop_id,
        stm.stop_sequence,
        stm.actual_arrival_pacific as tu_arrival_time,
        stm.n_predictions
    from stm
    left join sched
        on sched.service_date = stm.service_date
        and sched.trip_id = stm.trip_id
        and sched.base64_url = stm.schedule_base64_url
    order by trip_instance_key, stop_sequence
    """,
)

In [ ]:
tu_stop_times.head()

In [ ]:
# Get the scheduled stop times
# dim_stop_arrivals is partitioned by _feed_valid_from and clustered by feed_key,
# so both have to be literals here. We already have the feed keys, so dim_schedule_feeds
# is only needed for the matching _valid_from. One feed version can span several
# service dates, so this is fewer feeds than dates.
schedule_feed_keys = feed_keys_to_dates.loc[
    feed_keys_to_dates["date"].astype(str).isin(stm_dates), "schedule_feed_key"
].unique()
feed_keys_str = "('" + "', '".join(schedule_feed_keys) + "')"

schedule_feeds = sql.query_sql(
    f"""
    select
        key as feed_key,
        _valid_from as feed_valid_from
    from mart_gtfs.dim_schedule_feeds
    where key in {feed_keys_str}
    """,
)

valid_from_str = (
    "("
    + ", ".join(f"timestamp('{ts}')" for ts in schedule_feeds["feed_valid_from"].astype(str))
    + ")"
)

# dim_stop_arrivals is per feed version, not per date. fct_scheduled_trips expands
# a feed's trips out per service date and carries trip_instance_key, which is the
# key the VP stop arrivals use.
schedule_stop_times = sql.query_sql(
    f"""
    with sched_trips as (
        select trip_instance_key, feed_key, trip_id, service_date
        from mart_gtfs.fct_scheduled_trips
        where
            service_date in {stm_dates_str}
            and name = '{GTFS_DATASET_NAME}'
    ),
    arrivals as (
        select feed_key, trip_id, stop_id, stop_sequence, feed_timezone,
               arrival_sec, departure_sec
        from mart_gtfs.dim_stop_arrivals
        where
            _feed_valid_from in {valid_from_str}
            and feed_key in {feed_keys_str}
    )
    -- arrival_sec/departure_sec are gtfs time - count from midnight and are allowed to run past
    -- 24h on owl trips, so they are added to midnight in the feed's own timezone
    -- and converted to Pacific, matching how the VP and TU arrival times read.
    select
        sched_trips.trip_instance_key,
        sched_trips.service_date,
        arrivals.stop_id,
        arrivals.stop_sequence,
        datetime(
            timestamp_add(
                timestamp(sched_trips.service_date, arrivals.feed_timezone),
                interval arrivals.arrival_sec second
            ),
            'America/Los_Angeles'
        ) as schedule_arrival_time,
        datetime(
            timestamp_add(
                timestamp(sched_trips.service_date, arrivals.feed_timezone),
                interval arrivals.departure_sec second
            ),
            'America/Los_Angeles'
        ) as schedule_departure_time
    from arrivals
    inner join sched_trips
        on sched_trips.feed_key = arrivals.feed_key
        and sched_trips.trip_id = arrivals.trip_id
    order by trip_instance_key, stop_sequence
    """,
)
schedule_stop_times.head()


In [ ]:
# Get matching VP stop times

VP_COLS = ["trip_instance_key", "stop_sequence", "arrival_time", "stop_meters", "shape_array_key"]

# trip_instance_key is unique per trip per service date, so it both filters the
# statewide VP files down to this operator and carries the date back onto the
# joined rows. Pushing the filter into the read gets ~200k rows off GCS instead
# of the ~9.6M statewide rows these four files hold.
trip_dates = operator_trips.set_index("trip_instance_key")["service_date"]

vp_stop_times = g.read_parquet(
    stop_time_metrics_counts["vp_uri"].tolist(),
    columns=VP_COLS,
    filters=[("trip_instance_key", "in", set(trip_dates.index))],
).rename(columns={"arrival_time": "vp_arrival_time"})

vp_tu_stop_times = vp_stop_times.merge(
    tu_stop_times.drop(columns=["service_date"]),
    on=["trip_instance_key", "stop_sequence"],
    how="outer",
    indicator="source",
)
vp_tu_stop_times["source"] = vp_tu_stop_times["source"].cat.rename_categories(
    {"left_only": "vp_only", "right_only": "tu_only"}
)

# Outer again, so scheduled stops that neither VP nor TU ever reported survive.
vp_tu_stop_times = vp_tu_stop_times.merge(
    schedule_stop_times.drop(columns=["service_date"]),
    on=["trip_instance_key", "stop_sequence"],
    how="outer",
    indicator="has_schedule",
    suffixes=("", "_schedule"),
)
vp_tu_stop_times["has_schedule"] = vp_tu_stop_times["has_schedule"] != "left_only"
# The two stop_id columns agree wherever both are populated, so keep one.
vp_tu_stop_times["stop_id"] = vp_tu_stop_times["stop_id"].fillna(
    vp_tu_stop_times["stop_id_schedule"]
)
vp_tu_stop_times = vp_tu_stop_times.drop(columns=["stop_id_schedule"])
# Rows only the schedule knows about have no VP/TU source yet.
vp_tu_stop_times["source"] = (
    vp_tu_stop_times["source"].cat.add_categories("schedule_only").fillna("schedule_only")
)

vp_tu_stop_times["service_date"] = vp_tu_stop_times["trip_instance_key"].map(trip_dates)

print(
    vp_tu_stop_times.groupby(["service_date", "source"], observed=True)
    .size()
    .unstack(fill_value=0)
)
vp_tu_stop_times.head()


## Analysis

### Data availability

In [ ]:
# Hour of day distribution of stop time entries, stacked by service date
import matplotlib.pyplot as plt
import numpy as np
from calitp_data_analysis.calitp_color_palette import CALITP_CATEGORY_BOLD_COLORS

# Light mode, pinned explicitly so the figure does not inherit a dark notebook
# theme (a transparent background would put this near-black text on dark gray).
SURFACE = "#ffffff"
INK = "#0b0b0b"
INK_MUTED = "#52514e"
GRID = "#d9d9d9"
AXIS = "#b0b0b0"

# Slots 1, 2, 4, 3: green sits between orange and yellow so the two warmest
# hues are never adjacent segments within a stack.
DATE_COLORS = [CALITP_CATEGORY_BOLD_COLORS[i] for i in (0, 1, 3, 2)]

# grid position -> (source, which arrival time to bin, panel title)
PANELS = {
    (0, 0): ("both", "vp_arrival_time", "Matched (both) — VP arrival"),
    (0, 1): ("both", "tu_arrival_time", "Matched (both) — TU arrival"),
    (1, 0): ("vp_only", "vp_arrival_time", "VP only — VP arrival"),
    (1, 1): ("tu_only", "tu_arrival_time", "TU only — TU arrival"),
    (1, 2): ("schedule_only", "schedule_arrival_time", "Schedule only — scheduled arrival"),
}

service_dates = sorted(vp_tu_stop_times["service_date"].unique())
color_by_date = dict(zip(service_dates, DATE_COLORS))

# Rows share a y axis: the two "both" panels are the same matched rows counted
# two ways, and the bottom row is the three mutually exclusive misses. The rows
# differ in magnitude by ~5x, so their scales are left independent.
fig, axes = plt.subplots(
    2, 3, figsize=(18, 8), sharex=True, sharey="row", facecolor=SURFACE
)
fig.delaxes(axes[0, 2])  # only two ways to count a matched row

for (row, col), (src, time_col, title) in PANELS.items():
    ax = axes[row, col]
    subset = vp_tu_stop_times.loc[vp_tu_stop_times["source"] == src]
    hourly = (
        subset.groupby([subset[time_col].dt.hour, "service_date"], observed=True)
        .size()
        .unstack(fill_value=0)
        .reindex(index=range(24), columns=service_dates, fill_value=0)
    )

    bottom = np.zeros(24)
    for service_date in service_dates:
        heights = hourly[service_date].to_numpy()
        ax.bar(
            hourly.index,
            heights,
            bottom=bottom,
            width=0.82,
            color=color_by_date[service_date],
            label=str(service_date),
            edgecolor=SURFACE,
            linewidth=0.6,
        )
        bottom += heights

    ax.set_facecolor(SURFACE)
    ax.set_title(f"{title}  (n={len(subset):,})", fontsize=11, loc="left", color=INK)
    ax.set_xticks(range(0, 24, 2))
    ax.grid(axis="y", color=GRID, linewidth=0.6)
    ax.set_axisbelow(True)
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)
    for side in ("left", "bottom"):
        ax.spines[side].set_color(AXIS)
    ax.tick_params(colors=INK_MUTED, labelsize=9, labelbottom=True)

for ax in axes[:, 0]:
    ax.set_ylabel("stop time entries", fontsize=10, color=INK_MUTED)
for ax in axes[1, :]:
    ax.set_xlabel("hour of day (Pacific)", fontsize=10, color=INK_MUTED)

handles, labels = axes[0, 0].get_legend_handles_labels()
legend = fig.legend(
    handles,
    labels,
    title="service date",
    loc="lower center",
    ncol=len(service_dates),
    frameon=False,
    fontsize=9,
    title_fontsize=9,
)
legend.get_title().set_color(INK_MUTED)
for text in legend.get_texts():
    text.set_color(INK_MUTED)

fig.suptitle(
    "When VP, TU and scheduled stop times occur, by hour and service date",
    fontsize=13,
    x=0.5,
    y=0.98,
    color=INK,
)
fig.tight_layout(rect=[0, 0.05, 1, 0.96])
plt.show()
# this plot seems to show that VP only trips have a strong time pattern (mornings on all dates, evenings specifically on one date. 
# this matches hypothesis that TU is producing trips for certain dates only, because it runs on data that is partitioned by UTC date rather than service_date
# however, TU arrivals are occuring for all dates evenly. this suggests that there are significant data availability issues for the VP data, although
# could also infer that erroneous trips are occuring


In [ ]:
# TODO - would be interesting to know if missing VP trips are clustered on the same trips and/or same stops
# many clustered on same trip - means that vp is either consistently unavailable for some trips 
# or that trips are being dropped by the pipeline
# many clustered on same stop - issue with GPS/cellular outages at specific stops
# no clustering - VP pipeline is potentially dropping stops

### Basic difference analysis

In [ ]:
vp_tu_stop_times_cleaned = vp_tu_stop_times.loc[
    (
        (vp_tu_stop_times["service_date"] != datetime.date(2026, 4, 8))
        & (vp_tu_stop_times["source"] == "both")
    )
].copy()
vp_tu_stop_times_cleaned["vp_tu_difference"] = (
    vp_tu_stop_times_cleaned["vp_arrival_time"] - vp_tu_stop_times_cleaned["tu_arrival_time"]
).abs()
vp_tu_stop_times_cleaned["vp_tu_difference_seconds"] = vp_tu_stop_times_cleaned["vp_tu_difference"].dt.seconds
vp_tu_stop_times_cleaned.head()

In [ ]:
# Headline difference analysis
vp_tu_stop_times_grouped_by_date = vp_tu_stop_times_cleaned.groupby("service_date")["vp_tu_difference_seconds"]
diff_stats = pd.DataFrame(
    {
        "mean": vp_tu_stop_times_grouped_by_date.mean(),
        "p5": vp_tu_stop_times_grouped_by_date.quantile(0.05),
        "p25": vp_tu_stop_times_grouped_by_date.quantile(0.25),
        "p50": vp_tu_stop_times_grouped_by_date.quantile(0.50),
        "p75": vp_tu_stop_times_grouped_by_date.quantile(0.75),
        "p90": vp_tu_stop_times_grouped_by_date.quantile(0.90),
        "p95": vp_tu_stop_times_grouped_by_date.quantile(0.95),
        "std_dev": vp_tu_stop_times_grouped_by_date.std(),
    }
)
diff_stats.index.name = "service_date"
diff_stats.round(1)

In [ ]:
SMALL_HIST = 60
# The tail runs to ~85 min but 99.4% of rows are under 10 min, so the axis stops
MAX_SECONDS = 600
TICK_SECONDS = 120  # a multiple of the bin width that divides MAX_SECONDS evenly

diff_seconds = vp_tu_stop_times_cleaned["vp_tu_difference"].dt.total_seconds()
bins = np.arange(0, MAX_SECONDS + 2 * SMALL_HIST, SMALL_HIST)
within_noise = (diff_seconds < SMALL_HIST).mean()

# Reuse color_by_date so a date keeps the color it has in the hour of day plot.
# 2026-04-08 is filtered out of the cleaned frame; the remaining dates keep their
# own colors rather than being repainted.
cleaned_dates = [
    d for d in service_dates if d in set(vp_tu_stop_times_cleaned["service_date"])
]

fig, ax = plt.subplots(figsize=(12, 6), facecolor=SURFACE)
ax.hist(
    [
        diff_seconds[vp_tu_stop_times_cleaned["service_date"] == d].clip(upper=MAX_SECONDS)
        for d in cleaned_dates
    ],
    bins=bins,
    stacked=True,
    color=[color_by_date[d] for d in cleaned_dates],
    label=[str(d) for d in cleaned_dates],
    edgecolor=SURFACE,
    linewidth=0.6,
)

# The first bin is the noise floor; mark its right edge.
ax.axvline(SMALL_HIST, color=INK, linewidth=1.2, linestyle="--")
ax.annotate(
    f"{SMALL_HIST}s in the next chart \n{within_noise:.1%} within",
    xy=(SMALL_HIST, ax.get_ylim()[1]),
    xytext=(8, -22),
    textcoords="offset points",
    fontsize=9,
    color=INK,
)

ax.set_facecolor(SURFACE)
ax.set_title(
    f"VP vs TU arrival time difference  (n={len(diff_seconds):,})",
    fontsize=12,
    loc="left",
    color=INK,
)
ax.set_xlabel(
    f"|VP arrival − TU arrival| (seconds, {SMALL_HIST}s bins)",
    fontsize=10,
    color=INK_MUTED,
)
ax.set_ylabel("stop time entries", fontsize=10, color=INK_MUTED)
# The last tick sits on the overflow bin, so it gets a "+" label.
ticks = np.arange(0, MAX_SECONDS + TICK_SECONDS, TICK_SECONDS)
ax.set_xticks(ticks)
ax.set_xticklabels([f"{t:.0f}" for t in ticks[:-1]] + [f"{MAX_SECONDS}+"])
ax.grid(axis="y", color=GRID, linewidth=0.6)
ax.set_axisbelow(True)
for side in ("top", "right"):
    ax.spines[side].set_visible(False)
for side in ("left", "bottom"):
    ax.spines[side].set_color(AXIS)
ax.tick_params(colors=INK_MUTED, labelsize=9)

legend = ax.legend(title="service date", frameon=False, fontsize=9, title_fontsize=9)
legend.get_title().set_color(INK_MUTED)
for text in legend.get_texts():
    text.set_color(INK_MUTED)

fig.tight_layout()
plt.show()


In [ ]:
# Zoom in on the sub-minute differences to examine distribution there
SMALL_HIST_SECONDS = 5
CUTOFF_SECONDS = 60

diff_under_minute = diff_seconds[diff_seconds <= CUTOFF_SECONDS]
fine_bins = np.arange(0, CUTOFF_SECONDS + SMALL_HIST_SECONDS, SMALL_HIST_SECONDS)
cleaned_service_dates = vp_tu_stop_times_cleaned["service_date"]

fig, ax = plt.subplots(figsize=(12, 6), facecolor=SURFACE)
ax.hist(
    [
        diff_under_minute[cleaned_service_dates.loc[diff_under_minute.index] == d]
        for d in cleaned_dates
    ],
    bins=fine_bins,
    stacked=True,
    color=[color_by_date[d] for d in cleaned_dates],
    label=[str(d) for d in cleaned_dates],
    edgecolor=SURFACE,
    linewidth=0.6,
)

ax.set_facecolor(SURFACE)
ax.set_title(
    f"VP vs TU difference under {CUTOFF_SECONDS}s  "
    f"(n={len(diff_under_minute):,} of {len(diff_seconds):,})",
    fontsize=12,
    loc="left",
    color=INK,
)
ax.set_xlabel(
    f"|VP arrival − TU arrival| (seconds, {SMALL_HIST_SECONDS}s bins)",
    fontsize=10,
    color=INK_MUTED,
)
ax.set_ylabel("stop time entries", fontsize=10, color=INK_MUTED)
ax.set_xticks(np.arange(0, CUTOFF_SECONDS + 10, 10))
ax.grid(axis="y", color=GRID, linewidth=0.6)
ax.set_axisbelow(True)
for side in ("top", "right"):
    ax.spines[side].set_visible(False)
for side in ("left", "bottom"):
    ax.spines[side].set_color(AXIS)
ax.tick_params(colors=INK_MUTED, labelsize=9)

legend = ax.legend(title="service date", frameon=False, fontsize=9, title_fontsize=9)
legend.get_title().set_color(INK_MUTED)
for text in legend.get_texts():
    text.set_color(INK_MUTED)

fig.tight_layout()
plt.show()


### Outlier Investigation

In [ ]:
OUTLIER_DIFFERENCE = datetime.timedelta(seconds=120)
vp_tu_stop_times_outliers = vp_tu_stop_times_cleaned.loc[
    vp_tu_stop_times_cleaned["vp_tu_difference"] > OUTLIER_DIFFERENCE
].copy()
print(len(vp_tu_stop_times_outliers))

In [ ]:
print(len(vp_tu_stop_times_outliers) / len(vp_tu_stop_times_cleaned))

In [ ]:
vp_tu_stop_times_outliers.head()

In [ ]:
vp_tu_stop_times_outliers["vp_schedule_difference"] = (
    vp_tu_stop_times_outliers["schedule_arrival_time"] - vp_tu_stop_times_outliers["vp_arrival_time"]
).abs()
vp_tu_stop_times_outliers["tu_schedule_difference"] = (
    vp_tu_stop_times_outliers["schedule_arrival_time"] - vp_tu_stop_times_outliers["tu_arrival_time"]
).abs()


In [ ]:
# Spread of the schedule differences among the vp/tu outliers (signed, seconds)
OUTLIER_DIFF_COLS = ["vp_schedule_difference", "tu_schedule_difference"]
PERCENTILES = {"p5": 0.05, "p10": 0.1, "p15": 0.15, "p25": 0.25, "p50": 0.50, "p75": 0.75, "p90": 0.90, "p95": 0.95}

outlier_diff_seconds = vp_tu_stop_times_outliers[OUTLIER_DIFF_COLS].apply(
    lambda column: column.dt.total_seconds()
)
outlier_stats = pd.DataFrame(
    {
        "mean": outlier_diff_seconds.mean(),
        **{name: outlier_diff_seconds.quantile(q) for name, q in PERCENTILES.items()},
        "std_dev": outlier_diff_seconds.std(),
    }
)
outlier_stats.index.name = "metric"
outlier_stats.round(1)
# there isn't really a clear pattern here, 
# except the p5/p10 is interesting. seems likely that the schedule rt will "snap" to schedule times. 
# also TU times are all within 1 min of schedule times. this decreases the usefulness of TU

In [ ]:
# Jump detection: attach the previous 3 arrival times within each trip.
# trip_instance_key is the per trip per service date key, so grouping on it keeps
# one trip's last stops from leaking into the next trip's first stops.
# The frame already arrives in trip/stop order (both queries order by them and the
# outer merges sort on the join keys), so this asserts that rather than re-sorting.
LAG_STEPS = (1, 2, 3)
ARRIVAL_COLS = ["vp_arrival_time", "tu_arrival_time"]

trips = vp_tu_stop_times_cleaned.groupby("trip_instance_key", sort=False)
assert trips["stop_sequence"].is_monotonic_increasing.all(), "not in stop_sequence order"

vp_tu_stop_times_lagged = vp_tu_stop_times_cleaned.assign(
    **{
        f"{col}_lag{lag}": trips[col].shift(lag)
        for col in ARRIVAL_COLS
        for lag in LAG_STEPS
    }
)

vp_tu_stop_times_lagged[
    ["trip_instance_key", "stop_sequence", "source", "vp_arrival_time"]
    + [f"vp_arrival_time_lag{lag}" for lag in LAG_STEPS]
].head(10)


### Analyze backwards jumps

In [ ]:
# Jump size: how far each arrival moved relative to the 1-3 entries before it.
# Within a trip, arrival times should only ever increase with stop_sequence, so a
# negative jump is time running backwards - the "came back down" half of a jump.
jump_columns = {
    f"{col.removesuffix('_arrival_time')}_jump{lag}_seconds": (
        vp_tu_stop_times_lagged[col] - vp_tu_stop_times_lagged[f"{col}_lag{lag}"]
    ).dt.total_seconds()
    for col in ARRIVAL_COLS
    for lag in LAG_STEPS
}
vp_tu_stop_times_lagged = vp_tu_stop_times_lagged.assign(**jump_columns)
JUMP_COLS = list(jump_columns)

# How often does each feed run backwards? The denominator is entries that have a
# lag to compare against, so trip starts and gaps from the other feed drop out.
backwards_steps = pd.DataFrame(
    {
        "entries": vp_tu_stop_times_lagged[JUMP_COLS].notna().sum(),
        "backwards": (vp_tu_stop_times_lagged[JUMP_COLS] < 0).sum(),
    }
)
backwards_steps["pct_backwards"] = (
    backwards_steps["backwards"] / backwards_steps["entries"] * 100
).round(2)
# How far time runs backwards when it does, in seconds. Only the negative jumps
# are averaged, so this is the typical size of a backwards step rather than a
# net drift across all entries.
jumps = vp_tu_stop_times_lagged[JUMP_COLS]
backwards_steps["mean_backwards_time_difference_seconds"] = (
    jumps.where(jumps < 0).abs().mean().round(1)
)
backwards_steps.index.name = "metric"
backwards_steps
